Importing all the required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
train = pd.read_csv('train-test.csv')
validation = pd.read_csv('validation.csv')
train.head()

Performing EDA

In [ ]:
train.info()

In [ ]:
train.shape, validation.shape

In [ ]:
# validation has no posted_rate column
validation.head()

In [ ]:
train.isnull().sum()

In [ ]:
validation.isnull().sum()

We fill empty values with the median of the column

In [ ]:
weight_median = train['weight'].median()
market_index_median = train['market_index'].median()
print('weight median       :', weight_median)
print('market index median :', round(market_index_median, 4))

In [ ]:
train['weight'] = train['weight'].fillna(weight_median)
train['market_index'] = train['market_index'].fillna(market_index_median)

In [ ]:
train.duplicated().sum()

In [ ]:
train.describe()

Checking the weight column because the minimum is negative

In [ ]:
print('negative weights in train      :', (train['weight'] < 0).sum())
print('negative weights in validation :', (validation['weight'] < 0).sum())

We should take the absolute value as weights cannot be negative

In [ ]:
train['weight'] = train['weight'].abs()
validation['weight'] = validation['weight'].abs()

Checking the dates

In [ ]:
train['date'] = pd.to_datetime(train['date'])
validation['date'] = pd.to_datetime(validation['date'])

In [ ]:
print('train dates :', train['date'].min(), 'to', train['date'].max())
print('validation dates :', validation['date'].min(), 'to', validation['date'].max())

There is no date overlap.

This means we are predicting the future so need to split by date.

In [ ]:
# number of loads in each month
train['date'].dt.to_period('M').value_counts().sort_index()

Looking at the target column

In [ ]:
train['posted_rate'].describe()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(train['posted_rate'], bins=100)
plt.title('Distribution of posted rate')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(train[['distance', 'weight', 'market_index', 'quote_signal', 'posted_rate']].corr(), annot=True)
plt.title('Correlation with posted rate')
plt.show()

Distance has a 0.91 correlation with the rate and hides every other signal.

In trucking the normal way to compare prices is rate per mile, so lets make that column.

In [ ]:
train['rate_per_mile'] = train['posted_rate'] / train['distance']
train['rate_per_mile'].describe()

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(train[['distance', 'weight', 'market_index', 'quote_signal', 'rate_per_mile']].corr(), annot=True)
plt.title('Correlation with rate per mile')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(train['rate_per_mile'], bins=100)
plt.title('Distribution of rate per mile')
plt.show()

Most values sit around 2 but a few are very far away. Using a log scale to see them clearly.

In [ ]:
train['log_rate_per_mile'] = np.log(train['rate_per_mile'])

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(train['log_rate_per_mile'], bins=100)
plt.title('Log of rate per mile')
plt.show()

There are three separate distributions with empty gaps in between them.

Real prices would be one smooth distribution, so the two small side humps might be ignorable (bad data)

In [ ]:
# counting how many rows fall in each distribution set
print('low :', (train['log_rate_per_mile'] <= 0.4).sum())
print('main :', ((train['log_rate_per_mile'] > 0.4) & (train['log_rate_per_mile'] < 1.3)).sum())
print('high :', (train['log_rate_per_mile'] >= 1.3).sum())

In [ ]:
# marking the good rows
train['is_clean'] = (train['log_rate_per_mile'] > 0.4) & (train['log_rate_per_mile'] < 1.3)
train['is_clean'].value_counts()

In [ ]:
percent_bad = (train['is_clean'] == False).sum() / train.shape[0] * 100
print('bad rows percentage :', percent_bad)

Checking if the bad rows are grouped in some month or equipment, or spread randomly

In [ ]:
train[train['is_clean'] == False].groupby(train['date'].dt.to_period('M'))['is_clean'].value_counts()

Dropping the outliers

In [ ]:
train = train[train['is_clean']]

Checking the cities

In [ ]:
train_cities = set(train['pickup']) | set(train['delivery'])
validation_cities = set(validation['pickup']) | set(validation['delivery'])
print('cities in train      :', len(train_cities))
print('cities in validation :', len(validation_cities))

In [ ]:
extra_cities = validation_cities - train_cities
extra_cities

There are 8 cities in validation that are not in training data.

We should use latitude and longitude instead because those are numbers that work for any city.

#### Checking how the market index behaves over time

In [ ]:
market_per_day = train.groupby('date')['market_index'].mean()
plt.figure(figsize=(12, 6))
plt.plot(market_per_day)
plt.title('Market index over the year')
plt.show()

Now checking if the market index actually moves the price

In [ ]:
# comparing day by day instead of row by row, because single rows are noisy
daily = train.groupby('date').agg(rate_per_mile=('rate_per_mile', 'median'),
                                        market_index=('market_index', 'mean'))
daily.corr()

Daily rate per mile and daily market index have a high correlation of 0.64, market index is the time signal

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(x='market_index', y='rate_per_mile', data=daily)
plt.title('Daily market index vs daily rate per mile')
plt.show()

Checking the other columns against rate per mile

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(x='distance', y='rate_per_mile', data=train, alpha=0.3)
plt.title('Distance vs rate per mile')
plt.show()

In [ ]:
# check equipment rate differences
train.groupby('equipment')['rate_per_mile'].mean()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(x='equipment', y='rate_per_mile', data=train)
plt.title('Rate per mile by equipment')
plt.show()

In [ ]:
# heavier loads cost more per mile
train.groupby(pd.cut(train['weight'], [0, 15000, 25000, 32000, 40000, 50000]),
                    observed=True)['rate_per_mile'].mean()

In [ ]:
train.groupby('pickup')['rate_per_mile'].mean().sort_values()

In [ ]:
train.groupby('delivery')['rate_per_mile'].mean().sort_values()

Findings from the EDA

1. Validation is two months in the future so we must split by date, not randomly.
2. About 1.4 percent of the rows were noise, we should not use them from training.
3. Some weights are negative, we take the absolute value.
4. Validation has 8 new cities so we use latitude and longitude, not city names.
5. Distance, market index, weight, equipment and coordinates all affect the rate per mile.

Preparing the features

In [ ]:
def make_features(data):
    features = pd.DataFrame()
    features['distance'] = data['distance']
    features['market_index'] = data['market_index'].fillna(market_index_median)
    features['weight'] = data['weight'].fillna(weight_median)
    features['equipment'] = data['equipment'].map({'Dry Van': 0, 'Reefer': 1, 'Flatbed': 2})
    features['pickup_lat'] = data['pickup_lat']
    features['pickup_lon'] = data['pickup_lon']
    features['delivery_lat'] = data['delivery_lat']
    features['delivery_lon'] = data['delivery_lon']
    features['quote_signal'] = data['quote_signal']
    return features

We predict rate per mile instead of the rate itself, because dividing by distance removes the distance effect.

We multiply the prediction back by distance at the end to get the rate.

Splitting by date, we can train on the older months and test on the newest months

In [ ]:
train_part = train[train['date'] < '2025-09-01']
test_part = train[train['date'] >= '2025-09-01']

X_train = make_features(train_part)
y_train = train_part['rate_per_mile']
X_test = make_features(test_part)
y_test = test_part['rate_per_mile']

print('train rows :', len(train_part))
print('test rows  :', len(test_part))

print("train set percentage: ", len(train_part) / len(train) * 100)
print("test set percentage: ", len(test_part) / len(train) * 100)

Comparing a few models

In [ ]:
# turns the predicted rate per mile back into a rate and returns the error in percent
def get_mape(model, features, actual_rate, distance):
    predicted_rate = model.predict(features) * distance
    return np.mean(np.abs(predicted_rate - actual_rate) / actual_rate) * 100

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=36),
    'Random Forest': RandomForestRegressor(n_estimators=50, random_state=36),
    'Extra Trees': ExtraTreesRegressor(n_estimators=200, random_state=36),
    'Gradient Boosting': HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08,
                                                       max_leaf_nodes=127, random_state=36)
}

In [ ]:
mape_scores = []

for name, model in models.items():
    model.fit(X_train, y_train)
    mape_scores.append(get_mape(model, X_test, test_part['posted_rate'], test_part['distance']))
    print(name, 'is done')

performance = pd.DataFrame({'Model': models.keys(), 'MAPE': mape_scores}).sort_values('MAPE')
performance

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='MAPE', data=performance)
plt.xticks(rotation=30)
plt.title('Model comparison, lower MAPE is better')
plt.show()

The tree models are close to each other here but this is only one split.

Now also training and checking on more than one split

Comparing the top three models on four different date splits

In [ ]:
fold_results = []

for split_date in ['2025-07-01', '2025-08-01', '2025-09-01', '2025-10-01']:
    fold_train = train[train['date'] < split_date]
    fold_test = train[train['date'] >= split_date]

    scores = {'Test from': split_date}
    for name, model in [('Random Forest', RandomForestRegressor(n_estimators=100, random_state=36)),
                        ('Extra Trees', ExtraTreesRegressor(n_estimators=200, random_state=36)),
                        ('Gradient Boosting', HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08,
                                                                            max_leaf_nodes=127, random_state=36))]:
        model.fit(make_features(fold_train), fold_train['rate_per_mile'])
        scores[name] = get_mape(model, make_features(fold_test), fold_test['posted_rate'], fold_test['distance'])

    fold_results.append(scores)

fold_table = pd.DataFrame(fold_results)
fold_table

In [ ]:
fold_table[['Random Forest', 'Extra Trees', 'Gradient Boosting']].mean()

Extra Trees has the lowest error on every split, so we pick Extra Trees.

Checking that dropping the bad rows was actually worth it

In [ ]:
# reading the file again because our train already has the bad rows removed
dirty = pd.read_csv('train-test.csv')
dirty['date'] = pd.to_datetime(dirty['date'])
dirty['weight'] = dirty['weight'].abs()
dirty = dirty[dirty['date'] < '2025-09-01']

dirty_model = ExtraTreesRegressor(n_estimators=200, random_state=36)
dirty_model.fit(make_features(dirty), dirty['posted_rate'] / dirty['distance'])

print('MAPE with bad rows    :', round(get_mape(dirty_model, X_test, test_part['posted_rate'], test_part['distance']), 2))
print('MAPE without bad rows :', round(performance['MAPE'].min(), 2))

Training on the bad rows makes the error much worse, so the cleaning step was the most important one.

Trying a few sizes for the extra trees model

In [ ]:
tree_counts = [50, 100, 200, 300]
tree_mapes = []

for trees in tree_counts:
    model = ExtraTreesRegressor(n_estimators=trees, random_state=36)
    model.fit(X_train, y_train)
    tree_mapes.append(get_mape(model, X_test, test_part['posted_rate'], test_part['distance']))

pd.DataFrame({'Trees': tree_counts, 'MAPE': tree_mapes})

The score barely changes after 200 trees, so we use 200 and keep the model small.

Training the final model on all the clean data

In [ ]:
final_model = ExtraTreesRegressor(n_estimators=200, random_state=36)
final_model.fit(make_features(train), train['rate_per_mile'])

Predicting the validation loads

In [ ]:
validation_predicted = final_model.predict(make_features(validation)) * validation['distance']
validation_predicted.describe()

In [ ]:
# the scorer rejects zero or negative rates so checking that
(validation_predicted <= 0).sum()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(validation_predicted, bins=60)
plt.title('Predicted rates for the validation loads')
plt.show()

In [ ]:
predictions = pd.DataFrame({'load_id': validation['load_id'], 'predicted_rate': validation_predicted.round(2)})
predictions.head()

In [ ]:
# checking the ids match the template before saving
template = pd.read_csv('validation-predictions-template.csv')
print('same number of rows :', len(predictions) == len(template))
print('same ids            :', set(predictions['load_id']) == set(template['load_id']))

In [ ]:
predictions.to_csv('validation_predictions.csv', index=False)
predictions.shape

For December chart, every input is fixed and only the date changes

In [ ]:
december = pd.read_csv('december-chart-inputs.csv')
december['date'] = pd.to_datetime(december['date'])
december.head()

The december file has no market_index column, but the market index is our only time feature.

The validation file already covers December, so we can take the daily average from there.

In [ ]:
market_by_date = validation.groupby('date')['market_index'].mean()
december['market_index'] = december['date'].map(market_by_date)
december['market_index'].isnull().sum()

In [ ]:
# the december file has no coordinates so we take them from the training data
lexington = train[train['pickup'] == 'Lexington'][['pickup_lat', 'pickup_lon']].iloc[0]
fort_wayne = train[train['delivery'] == 'Fort Wayne'][['delivery_lat', 'delivery_lon']].iloc[0]
lexington, fort_wayne

In [ ]:
december['pickup_lat'] = lexington['pickup_lat']
december['pickup_lon'] = lexington['pickup_lon']
december['delivery_lat'] = fort_wayne['delivery_lat']
december['delivery_lon'] = fort_wayne['delivery_lon']

In [ ]:
# quote_signal is not given either so we can use the training average
december['quote_signal'] = train['quote_signal'].mean()

In [ ]:
december['predicted_rate'] = (final_model.predict(make_features(december)) *
                              december['distance']).round(2)
december[['date', 'market_index', 'predicted_rate']]

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(december['date'], december['predicted_rate'], marker='o')
plt.title('December predicted rate, Lexington to Fort Wayne')
plt.xticks(rotation=45)
plt.show()

The line goes up and down every week, which matches the weekly pattern we found in the market index earlier.

This makes sense because the date is the only thing changing in these 31 rows.

In [ ]:
# dropping the new columns added
original_columns = ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']
december_to_save = december[original_columns].copy()
december_to_save['date'] = december_to_save['date'].dt.strftime('%Y-%m-%d')
december_to_save.head()

In [ ]:
december_to_save.to_csv('december-chart-inputs.csv', index=False)
december_to_save.shape

Both files are saved now, so we can run the scorer

python score.py --predictions validation_predictions.csv --december-predictions december-chart-inputs.csv

Saving the model so we can use it later without training again

In [ ]:
import pickle

pickle.dump(final_model, open('model.pkl', 'wb'))

In [ ]:
# checking the saved model gives the same predictions
saved_model = pickle.load(open('model.pkl', 'rb'))
saved_predicted = saved_model.predict(make_features(december)) * december['distance']

print('same predictions :', np.allclose(saved_predicted, december['predicted_rate'], atol=0.01))